# BERTurk v3: foreign client names (free GPU)

Runs on **Kaggle** (Notebook → Settings → Accelerator **GPU T4 x2** or **P100**, **Internet on**; Kaggle asks for a
phone check once) or **Google Colab** (Runtime → Change runtime type → **T4 GPU**). Nothing runs on your laptop.

What it does, top to bottom (about 45–60 minutes on a T4):

1. gets the code (branch `foreign-names`) and builds the foreign-client sets (held-out names never seen in training)
2. downloads the published **BERTurk v2** (`f1k/katip-tr-pii-berturk`) and scores it: that's the baseline
3. trains **BERTurk v3** = v2's recipe (synthetic + Turkish Wiki NER real text) **+ foreign clients**
4. scores v3 on the same sets, with a paired test against v2, plus the legal and Wiki checks so we see any regression
5. zips the model (and optionally uploads it to Hugging Face, if you add an `HF_TOKEN` secret)

Only public data is used: Katip's synthetic documents, Turkish Wiki NER (CC BY-SA 4.0), TurkishLegalNER (Apache-2.0)
and generated foreign names (Faker, MIT; curated lists, CC0). No client document is involved.

In [ ]:
import os, subprocess, sys
BRANCH = "foreign-names"
if not os.path.exists("katip"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/fikrifirat/katip"], check=True)
os.chdir("katip")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faker", "onnx", "onnxruntime", "regex", "tokenizers", "huggingface_hub"], check=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the accelerator on")

## 1 · Foreign-client sets and training mix

In [ ]:
!python lab/foreign/build.py
!python lab/foreign/mix.py --berturk

## 2 · Baseline: the published BERTurk v2

In [ ]:
from huggingface_hub import snapshot_download
V2 = snapshot_download("f1k/katip-tr-pii-berturk", local_dir="runs/berturk_v2")
print(V2, os.listdir(V2))

## 3 · Train BERTurk v3 (v2 recipe + foreign clients)

In [ ]:
!cd ml/berturk && python train_berturk.py --real-text --extra ../../lab/data/foreign/train_berturk_foreign.jsonl --out ../../runs/berturk_v3 --device cuda

## 4 · Scores: v2 vs v3

`score.py` compares the two model folders as a pair (v2 first = control). The headline is **foreign names in real
Yargıtay sentences** (court-foreign), with a 95% confidence interval and a McNemar test on the same name slots.
`score_legal.py` and the blind set check that Turkish names did not get worse.

In [ ]:
!python lab/foreign/score.py runs/berturk_v2:0.5 runs/berturk_v3:0.5 --tag berturk
!python lab/score_legal.py runs/berturk_v2:0.5 runs/berturk_v3:0.5

## 5 · Save the model (and the results)

In [ ]:
import shutil, json
shutil.make_archive("/kaggle/working/berturk_v3" if os.path.exists("/kaggle/working") else "berturk_v3", "zip", "runs/berturk_v3")
for f in ("lab/results/foreign_berturk.json", "lab/results/legal.json"):
    if os.path.exists(f): shutil.copy(f, "/kaggle/working/" if os.path.exists("/kaggle/working") else ".")
print("download berturk_v3.zip and foreign_berturk.json from the Output tab")

In [ ]:
# Optional: upload to Hugging Face as a new revision. Add a secret named HF_TOKEN (Kaggle: Add-ons → Secrets;
# Colab: the key icon). Leave PUSH = False to skip.
PUSH = False
REPO = "f1k/katip-tr-pii-berturk"
if PUSH:
    try:
        from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")
    except ImportError:
        from google.colab import userdata; token = userdata.get("HF_TOKEN")
    from huggingface_hub import HfApi
    HfApi(token=token).upload_folder(repo_id=REPO, folder_path="runs/berturk_v3", repo_type="model",
                                     commit_message="v3: + foreign client names (lab/foreign)", create_pr=True)
    print("opened a PR on", REPO, "- merge it on the Hub when the scores look right")